In [1]:
import sys
sys.path.append('..')

import pickle
import numpy as np
import pandas as pd

from utils.general_tools import pd_to_param_holder
from utils.data_processing_tools import full_preprocessing_pipeline, bring_to_original_range
from utils.modeling_tools import perform_modeling_linear
from utils.regression_tools import forecast_test_linear
from utils.display_tools import load_best_forecasts


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
lot = pickle.load(open("../saves_and_results/cache_lot.p", "rb"))
data = pickle.load(open("../saves_and_results/cache_endog.p", "rb"))
exogs = pickle.load(open("../saves_and_results/cache_exogs.p", "rb"))

In [4]:
loc = "Moehne"
direction = "lot"
d = lot[loc][direction]
y_ = d.loc[d.index.year == 2020]
# We can remove sarimax here since it is actually not relevant for the baseline. 
# (Collapses to linear model with bad estimation i guess.)

In [8]:
ms = {}

for x in [str(n) for n in range(0,4)]:
    model_spec, forecasts = load_best_forecasts("results/journal/" + x, direction="lot")
    if (x == "1")  or (x == "2"):
        forecasts.drop(columns="sarimax", inplace=True)
    score = pd.DataFrame(
        (np.abs(forecasts - y_.values).mean().values),
        index=forecasts.columns,
        columns=["MAE"],
    )
    ms[x] = model_spec

In [9]:
ms["1"][ms["1"].columns[4:-5]]

,P,D,Q,F,STAU,T,M_Stau,M_T,Decompose,Interaction,Estimator,n_estimator,max_depth,learning_rate
linear,0,NaN,NaN,0,[0],[0],0,0,0,0,NaN,NaN,NaN,NaN
forest,0,NaN,NaN,0,[0],[0],0,0,0,0,NaN,250.0,7.0,NaN
ada,0,NaN,NaN,0,[0],[0],0,0,0,0,linear,50.0,NaN,0.1
sarimax,0,0.0,0.0,0,[0],[0],0,0,0,0,NaN,NaN,NaN,NaN


In [10]:
ms["0"][ms["0"].columns[4:-5]]

,P,D,Q,F,STAU,T,M_Stau,M_T,Decompose,Interaction,Estimator,n_estimator,max_depth,learning_rate
linear,1,NaN,NaN,0,[0 1 2],[0 1 2],0,7,0,2,NaN,NaN,NaN,NaN
forest,1,NaN,NaN,0,[],[0 1 2],0,7,0,0,NaN,250.0,NaN,NaN
ada,1,NaN,NaN,0,[0 1],[0 1 2],0,0,0,2,linear,250.0,NaN,1.0
sarimax,0,1.0,2.0,0,[0 1],[0 1 2],7,7,1,0,NaN,NaN,NaN,NaN


In [11]:
# some ugly refractor because of loading: 
for y in ["T", "STAU"]:
    stack = []
    for x in ms["0"].index:
        split = ms["0"].loc[x,  y][1:-1]
        if len (split) != 0:
            stack.append(np.array(split.split(" ")).astype(int))
        else:
            stack.append(np.array([]))
   
    ms["0"][y] = stack

In [12]:
ms["0"]["Index"] = ms["0"].index
combo = pd_to_param_holder(ms["0"], "linear")

In [13]:
(train,
test,
max_data,
min_data,
trend,
seasonality,
test_trend,
test_seasonality) = (
full_preprocessing_pipeline(lot["Moehne"]["lot"], exogs["Moehne"], combo, model_search=False)
)

/home/stein/project_repos/dam/full_baseline/forecasting_analysis/utils/data_processing_tools.py:231: FutureWarning: Dropping of nuisance columns in rolling operations is deprecated; in a future version this will raise TypeError. Select only valid columns before calling the operation. Dropped columns were Index(['datetime'], dtype='object')
  exog_stau = exog.rolling(mean_period[1], closed="left").mean()


In [14]:
combo.export()

[1,
 nan,
 nan,
 0,
 [array([0, 1, 2]), array([0, 1, 2])],
 [0, 7],
 0,
 2,
 nan,
 nan,
 nan,
 nan,
 {}]

In [15]:
model, trend_step = perform_modeling_linear(train[0]["Radial"], train[1], combo)
pred = forecast_test_linear(test[0],test[1], combo, model, trend_step)

In [16]:
p = "Radial"
Y, Y_hat = bring_to_original_range(
    test[0][p],
    pred[0],
    max_data[p],
    min_data[p],
    (test_trend[p] if isinstance(trend, pd.DataFrame) else None),
    (test_seasonality[p] if isinstance(seasonality, pd.DataFrame) else None),
    lot["Moehne"]["lot"],
    p
)

In [17]:
np.abs(Y_hat.values- y_.values[:,0]).mean()

0.04570325571303048

In [18]:
names = perform_modeling_linear(train[0]["Radial"], train[1], combo, return_temporary=True).columns
final = model.summary().tables[1]
pd.DataFrame(final.data[1:], index = names)

,0,1,2,3,4,5,6
T_mean_7_0,x1,0.0215,0.004,4.869,0.000,0.013,0.030
W_0,x2,0.4153,0.091,4.552,0.000,0.236,0.594
W_1,x3,-0.4828,0.175,-2.759,0.006,-0.826,-0.140
W_2,x4,0.0906,0.091,0.993,0.321,-0.088,0.270
T_0,x5,0.3236,0.082,3.933,0.000,0.162,0.485
T_1,x6,-0.0568,0.006,-9.576,0.000,-0.068,-0.045
T_2,x7,0.0685,0.005,13.313,0.000,0.058,0.079
INTER,x8,-0.4160,0.083,-5.038,0.000,-0.578,-0.254
1,x9,0.9663,0.003,344.701,0.000,0.961,0.972
const,const,0.0305,0.002,14.525,0.000,0.026,0.035
